In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")

# spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "50000000")

This notebook will be ran in a job once a week to refresh the KPF Dashboard, only for TDC and MCP SSE campaigns.
- Offsite will need to be loaded manually and ran in the Closed Loop Absolute Notebook.
- Reminder SSE PUSH campaigns are not as frequent - more reasonable to also run manually as Wrap Reports are verified.

In [0]:
# Pre-pivoted closed loop data pulled from closed_loop_campaign_summary notebook
closed_loop_prepivot = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')
closed_loop_prepivot = closed_loop_prepivot.filter(f.col('camp_start_date') >= '2023-01-01')
closed_loop_prepivot.display()

# Metadata pulls from KPM
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')

### Absolute iROAS Calculation - TDC
- TO DO: Only get campaigns that are new, as in not in closed loop already 

In [0]:
# All 2024 - 2026 TDCs that are currently processed in absolute methodology
abs_emod_tdc = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc')
abs_emod_tdc.display()

In [0]:
# List of TDCs, dashboard already filters to Top Performer (%800% Product Group) and All Modalities
closed_loop_tdc_2026 = closed_loop_prepivot.filter(
    ((f.col("campaign_type") == "TDC") | (f.col("campaign_type") == "EMOD")) &
    (f.year(f.col("camp_start_date")) == 2026)
)

# Edge Case in MHTV addressed here (XCM labaled tdc)
kpf_dashboard_revamped_tdc_pre = closed_loop_tdc_2026.filter(
    (f.col('campaign_id') != 120767) &
    (~f.col('project_name').contains("XCM"))
)

# 3. Use a Left Anti Join to exclude anything already processed in abs_emod_tdc
# Matching on campaign_id ensures you catch campaigns even if the project name mutated slightly
kpf_dashboard_revamped_tdc = kpf_dashboard_revamped_tdc_pre.join(
    abs_emod_tdc, 
    on=['campaign_id'], 
    how='leftanti'
)

# 4. Display the new 2026 TDCs. Df will be empty if tdc is up-to date w/ abs numbers.
kpf_dashboard_revamped_tdc.display()

In [0]:
# Offer data (Project Id, coupon barcode, redemption barcode, effective date, expiration date)
mmoi_metadata = (mmoi
    .join(kpf_dashboard_revamped_tdc.select('kpm_duplicated_id').distinct(), mmoi.KPM_PROJECT_ID == kpf_dashboard_revamped_tdc.kpm_duplicated_id, 'inner')
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        f.first('COUPON_BARCODE').alias('coupon_barcode'),
        f.first('redemption_barcode').alias('redemption_barcode'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

# Should be only NEW TDCs that dont exist in dashboard
mmoi_metadata.display()

In [0]:
# Add Working Cost Column. TDC multiplier is 0.141, EMOD multiplier is 0.015
kpf_dashboard_revamped_tdc = (kpf_dashboard_revamped_tdc
    .withColumn(
        'working_cost',
        f.when(f.col('campaign_type') == 'EMOD', (f.col('camp_cost') * 0.015))
         .otherwise(f.col('camp_cost') * 0.141)
         .cast('double')
    )
)

# Join mmci (data already exists in dashboard revamped) with mmoi barcode/date information
kpf_dashboard_revamped_mmoi_tdc = (kpf_dashboard_revamped_tdc
    .join(mmoi_metadata, f.col('kpm_duplicated_id') == f.col('KPM_PROJECT_ID'), how='inner')
    .drop('KPM_PROJECT_ID')
)

kpf_dashboard_revamped_mmoi_tdc.display()

In [0]:
th_filter = (target_history
    .filter(f.col('test_control_id') == '1') # Filter early to reduce volume
    .withColumnRenamed('HSHD_CODE', 'ehhn')
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_tdc.select('target_id', 'campaign_id', 'kpm_duplicated_id', 'project_name')),
        on='target_id', 
        how='inner' # This automatically drops any target_ids not in your dashboard
    )
)

th_filter.display()

In [0]:
# Prepare necessary Points Detail - Faster Ver

points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))

# Select only columns needed from mmoi_mhtv_mmci_sse
kpf_dashboard_revamped_mmoi_tdc_subset = (kpf_dashboard_revamped_mmoi_tdc
    .select(
        'campaign_id', 
        'coupon_barcode', 
        'redemption_barcode', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_tdc_subset),
        on=points_detail_target_history.kpm_duplicated_id == kpf_dashboard_revamped_mmoi_tdc_subset.campaign_id,
        how="inner"
    )
    .filter(
        (f.col('offer') == f.col('coupon_barcode')) | 
        (f.col('offer') == f.col('redemption_barcode'))
    )
)

points_detail_target_history_offer_info = points_detail_target_history_offer_info.dropDuplicates()
points_detail_target_history_offer_info.display()

In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (
    points_detail_target_history_offer_info
    .filter(
        (f.col('trn_dt') >= f.to_date(f.col('effective_date'), 'yyyyMMdd')) &
        (f.col('trn_dt') <= f.to_date(f.col('expiration_date'), 'yyyyMMdd'))
    )
)

points_detail_target_history_filtered_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_filtered_offer_info
    .groupBy('kpm_duplicated_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.approx_count_distinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_tdc_df = (
    kpf_dashboard_revamped_mmoi_tdc
    .join(aggregated_df, on=['kpm_duplicated_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('double'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('double'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Added Redemption Cost explicitly to better match Shumaila's #s (same as cost_total_points_earned)
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_tdc_df.display()

# kpm_duplicated_id, project_name, campaign_id, pg_master, job_id, campaign_type, abs_aroas, adj_aroas, abs_iroas, adj_iroas, abs_total_cost, adj_total_cost, abs_sales_test_earned, abs_sales_uplift_earned, adj_sales_total, adj_sales_uplift, cost_total_points_redeemed, cost_total_points_earned,total_points_earned,

In [0]:
final_result_tdc_df.select(
    "kpm_duplicated_id", "project_name", "campaign_id", "pg_master", "job_id", "campaign_type",
    "abs_aroas", "adj_aroas", "abs_iroas", "adj_iroas", "abs_total_cost", "adj_total_cost",
    "abs_sales_test_earned", "abs_sales_uplift_earned", "adj_sales_total", "adj_sales_uplift",
    "cost_total_points_redemeed", "cost_total_points_earned", "total_points_earned", "working_cost", "redemption_cost"
).coalesce(1).write.mode("append").parquet(
    'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc'
)

# final_result_tdc_df_test = spark.read.parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc')
# final_result_tdc_df_test.display()

### Absolute iROAS Calculation - SSE

In [0]:
abs_mcp_sse = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')
abs_mcp_sse.display()

In [0]:
# List of SSEs, dashboard already filters to Top Performer (%800% Product Group) and All Modalities
kpf_dashboard_revamped_sse_pre = closed_loop_prepivot.filter(
  (f.col("campaign_type") == "SSE") &
  (f.year(f.col("camp_start_date")) == 2026)
)

# excluding all campaigns that already exist in the dash
kpf_dashboard_revamped_sse = kpf_dashboard_revamped_sse_pre.join(
    abs_mcp_sse, 
    on=['campaign_id'], 
    how='leftanti'
)

#campaign_id_list = [row['campaign_id'] for row in kpf_dashboard_revamped_sse.select('campaign_id').distinct().collect()]

kpf_dashboard_revamped_sse.display()


In [0]:
# Offer data (Project Id, coupon barcode, redemption barcode, effective date, expiration date)
mmoi_metadata = (mmoi
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        #f.first('COUPON_BARCODE').alias('coupon_barcodes'),
        #f.first('redemption_barcode').alias('redemption_barcodes'),
        f.collect_set('COUPON_BARCODE').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

mmoi_metadata.display()

In [0]:
# Calculate working cost, iroas (sales uplift / campaign cost), aroas (sales test total / campaign cost)
mhtv_metrics_agg = (kpf_dashboard_revamped_sse
    .groupBy('campaign_id')
    .agg(
        f.sum('sales_uplift_total').alias('sales_uplift_total'),
        f.sum('sales_test_total').alias('sales_test_total'),
        f.sum('camp_cost').alias('camp_cost')
    )
    .withColumn('working_cost', f.col('camp_cost') * 0.3390)
    .withColumn('iroas_original', f.round(f.col('sales_uplift_total') / f.col('camp_cost'), 2))
    .withColumn('aroas_original', f.round(f.col('sales_test_total') / f.col('camp_cost'), 2))
)

mhtv_metrics_agg.display()

In [0]:
# Join metadata 
mmoi_mhtv_sse = (mmoi_metadata
    .join(mhtv_metrics_agg, mmoi_metadata["KPM_PROJECT_ID"] == mhtv_metrics_agg["campaign_id"], how="right")
)

mmoi_mhtv_mmci_sse = (mmoi_mhtv_sse
    .join(mmci.select('kpm_project_id', 'target_id', 'project_name'), on = "KPM_PROJECT_ID", how = "inner")
)

mmoi_mhtv_mmci_sse.display()

In [0]:
target_ids_to_keep = [
    row['target_id'] for row in mmoi_mhtv_mmci_sse.select('target_id').distinct().collect()
]

# Reduce Target history table with only relevant target ids
test_hhs_slim = (
    target_history
    .filter(f.col('test_control_id') == '1')
    .filter(f.col('target_id').isin(target_ids_to_keep)) 
    .select(f.col('HSHD_CODE').alias('ehhn'), 'target_id')
    .distinct()
)

# Join Target history with metadata
th_filter = (
    test_hhs_slim.join(
        f.broadcast(mmoi_mhtv_mmci_sse.select('kpm_project_id', 'project_name', 'target_id').distinct()), 
        on='target_id', 
        how='inner'
    )
)

th_filter.display()

In [0]:
# Join target history + metadata with points detail
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))
display(points_detail_target_history)

In [0]:
# Select only columns needed from mmoi_mhtv_mmci_sse
mmoi_mhtv_mmci_sse_filtered = (mmoi_mhtv_mmci_sse
    .select(
        'kpm_project_id', 
        'coupon_barcodes', 
        'redemption_barcodes', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(mmoi_mhtv_mmci_sse_filtered),
        on=[
            points_detail_target_history.kpm_project_id == mmoi_mhtv_mmci_sse_filtered.kpm_project_id,
            (f.array_contains(mmoi_mhtv_mmci_sse_filtered.coupon_barcodes, points_detail_target_history.offer) | 
             f.array_contains(mmoi_mhtv_mmci_sse_filtered.redemption_barcodes, points_detail_target_history.offer))
            #points_detail_target_history.trn_dt.between(mmoi_mhtv_mmci_sse_filtered.effective_date, mmoi_mhtv_mmci_sse_filtered.expiration_date)
        ],
        how="inner"
    ).drop(mmoi_mhtv_mmci_sse_filtered.kpm_project_id)
)

points_detail_target_history_offer_info.display()

In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (points_detail_target_history_offer_info
    .filter(f.col('trn_dt').between('effective_date', 'expiration_date'))
)

points_detail_target_history_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_offer_info
    .groupBy('kpm_project_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.countDistinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_sse_df = (
    mmoi_mhtv_mmci_sse
    .join(aggregated_df, on=['kpm_project_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('Integer'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Adj Numbers - Redemption Cost to better match Shumaila's Numbers
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_sse_df.display()

In [0]:
# Dropping array columns (barcodes) to support csv and parquet writing
final_result_sse_df_barcodes_dropped = final_result_sse_df.drop('coupon_barcodes', 'redemption_barcodes')
final_result_sse_df_barcodes_dropped.coalesce(1).write.mode("append").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')

#final_result_sse_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')
#final_result_sse_df_test.display()